# PYNQ-Z1 MNIST validation · 64-32

Validate the **784 → 64 → 32 → 10** MLP with the ten MNIST batches (10,000 images). The notebook programs the FPGA and sends one image at a time through AXI DMA. Run one topology notebook at a time.

Keep `mlp_64_32.bit` and its matching `mlp_64_32.hwh` together in `topologies/64-32/`. Keep the matching `mlp_64_32.tcl` with the Vivado export. Open this notebook from the directory containing `pynq_validation.py`.

In [ ]:
from pathlib import Path

from pynq_validation import display_report, validate_on_board

TOPOLOGY = "64-32"
TOPOLOGIES_DIR = Path("/home/xilinx/vivado_projects/topologies")
TEST_DIR = Path("/home/xilinx/vivado_projects/test_files")
DMA_TIMEOUT_S = 2.0
BITSTREAM = None  # Set to the .bit path if the board uses another filename.

## Program the overlay and classify all batches

Edit the paths above if the board stores the artifacts elsewhere. If you set `BITSTREAM`, the selected file must have its matching `.hwh` beside it. Use only the overlay for this notebook's topology. Each pair of `test_samples_XX.mem` and `test_labels_XX.mem` files contains 1,000 images and their matching labels. The input words are already in the accelerator's fixed-point format. The predicted class is the index of the largest signed 32-bit output score.

In [ ]:
report = validate_on_board(
    TOPOLOGY,
    TOPOLOGIES_DIR,
    TEST_DIR,
    bitstream=BITSTREAM,
    timeout_s=DMA_TIMEOUT_S,
)

## Accuracy and classification time

- **Batch time** includes preparing inference buffers, DMA transfers and choosing each predicted class. It excludes file parsing, FPGA programming and warm-up. Throughput is images divided by batch time.
- **DMA latency** is the per-image round trip from DMA submission through completion, including accelerator execution and DMA software overhead.
- **Confusion matrix** rows are true digits and columns are predicted digits. Per-class accuracy is the diagonal count divided by the corresponding row total.

The full results are available in `report`. After a DMA timeout or transfer error, rerun from the top so the FPGA is reprogrammed. Reboot the board before retrying if the DMA reset failed.

In [ ]:
display_report(report)